In [1]:

import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import densenet121
from sklearn.model_selection import train_test_split


# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224,224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"   
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15,
                                           random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3,
                                         random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")



#  Load pretrained DenseNet121 FP32

ckpt_path = "/home/h5/sapi279d/densenet121_weights.pth"  # update path
state_dict = torch.load(ckpt_path, map_location="cpu")

# strip `_orig_mod.` prefix if present
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

num_classes = len(dataset.classes)  # 200 for your dataset
model_fp32 = densenet121(weights=None)
model_fp32.classifier = nn.Linear(model_fp32.classifier.in_features, num_classes)
model_fp32.load_state_dict(new_state_dict, strict=True)
model_fp32.eval()
print("FP32 DenseNet121 loaded")



# Power-of-Two Quantization function

def power_of_two_quantize(weight, n_bits=8):
    w = weight.clone()
    w_min, w_max = w.min(), w.max()
    if w_min == w_max:
        return w  # no quantization if weights are constant
    # scale based on range
    scale = (w_max - w_min) / (2 ** n_bits - 1)
    # snap scale to nearest power of two
    pow2_scale = 2 ** torch.round(torch.log2(scale))
    zero_point = torch.round(-w_min / pow2_scale)
    w_q = torch.round(w / pow2_scale + zero_point)
    w_q = torch.clamp(w_q, 0, 2 ** n_bits - 1)
    quantized_weight = pow2_scale * (w_q - zero_point)
    return quantized_weight



# Apply Power-of-Two Quantization

model_pot = densenet121(weights=None)
model_pot.classifier = nn.Linear(model_pot.classifier.in_features, num_classes)
model_pot.load_state_dict(new_state_dict, strict=True)

for name, param in model_pot.named_parameters():
    if "weight" in name:
        with torch.no_grad():
            param.copy_(power_of_two_quantize(param.data, n_bits=8))

model_pot.eval()
print("Applied Power-of-Two Quantization")



# Evaluation

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_pot  = evaluate_model(model_pot,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"Power-of-Two Quantized Accuracy: {acc_pot:.2f}%")



# Model size & Runtime

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    ms_per_batch = (total_time / num_batches) * 1000
    ms_per_image = (total_time / total_images) * 1000
    return ms_per_batch, ms_per_image

fp32_size = get_model_size(model_fp32)
pot_size  = get_model_size(model_pot)
print(f"FP32 size: {fp32_size:.2f} MB | Power-of-Two size: {pot_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_pot,  img_ms_pot  = benchmark(model_pot,  test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"Power-of-Two Inference: {batch_ms_pot:.2f} ms/batch | {img_ms_pot:.2f} ms/image")



✅ DataLoaders ready
✅ FP32 DenseNet121 loaded
✅ Applied Power-of-Two Quantization
FP32 Accuracy: 49.44%
Power-of-Two Quantized Accuracy: 50.78%
FP32 size: 29.19 MB | Power-of-Two size: 29.19 MB
FP32 Inference: 5166.07 ms/batch | 40.36 ms/image
Power-of-Two Inference: 5068.25 ms/batch | 39.60 ms/image
